In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import drive


drive.mount('/content/drive')

# =======================
# 0) PATHS
# =======================
BASE_DIR  = "/content/drive/MyDrive/Flood_Analysis"
LS_FILE   = os.path.join(BASE_DIR, "guinea_ls.csv")
RAIN_FILE = os.path.join(BASE_DIR, "guinea_era5land_dailyrain.csv")
OUT_FILE  = os.path.join(BASE_DIR, "final_merged_dataset_rainValidCells.csv")

print("Loading files...")
ls_df   = pd.read_csv(LS_FILE)
rain_df = pd.read_csv(RAIN_FILE)

print("Landsat shape:", ls_df.shape)
print("Rain shape:", rain_df.shape)

# =======================
# 1) STANDARDIZE LS
# =======================
ls_df.columns = [c.strip() for c in ls_df.columns]

# Standardize key column names to: Date, Region_ID, ValidPixCount
# (handles common variants safely)
for c in ls_df.columns:
    cl = c.lower().strip()
    if cl == "date":
        ls_df = ls_df.rename(columns={c: "Date"})
    elif cl in ["region_id", "regionid", "region"]:
        ls_df = ls_df.rename(columns={c: "Region_ID"})
    elif cl in ["validpixcount", "valid_pix_count", "valid_pixel_count"]:
        ls_df = ls_df.rename(columns={c: "ValidPixCount"})

required_ls = ["Date", "Region_ID", "ValidPixCount"]
missing_ls = [c for c in required_ls if c not in ls_df.columns]
if missing_ls:
    raise ValueError(f"LS file is missing required columns: {missing_ls}. "
                     f"Found columns: {ls_df.columns.tolist()}")

# Parse / cast
ls_df["Date"] = pd.to_datetime(ls_df["Date"], errors="coerce")
ls_df["Region_ID"] = pd.to_numeric(ls_df["Region_ID"], errors="coerce")
ls_df["ValidPixCount"] = pd.to_numeric(ls_df["ValidPixCount"], errors="coerce").fillna(0)

ls_df = ls_df.dropna(subset=["Date", "Region_ID"]).copy()
ls_df["Region_ID"] = ls_df["Region_ID"].astype(int)


before = len(ls_df)
ls_df = ls_df[ls_df["ValidPixCount"] > 0].copy()
print(f"✅ Fix A applied: dropped {before - len(ls_df)} rows (ValidPixCount<=0). Remaining: {len(ls_df)}")

# =======================
# 2) STANDARDIZE RAIN SCHEMA
# =======================
rain_df.columns = [c.strip().lower() for c in rain_df.columns]

# Canonical rename -> Region_ID, Date, Precip_mm
rename_map = {}
if "region_id" in rain_df.columns:
    rename_map["region_id"] = "Region_ID"
elif "regionid" in rain_df.columns:
    rename_map["regionid"] = "Region_ID"

if "date" in rain_df.columns:
    rename_map["date"] = "Date"

if "mean" in rain_df.columns:
    rename_map["mean"] = "Precip_mm"
elif "precip_mm" in rain_df.columns:
    rename_map["precip_mm"] = "Precip_mm"

rain_df = rain_df.rename(columns=rename_map)

required_rain = ["Region_ID", "Date", "Precip_mm"]
missing_rain = [c for c in required_rain if c not in rain_df.columns]
if missing_rain:
    raise ValueError(f"Rain file is missing required columns: {missing_rain}. "
                     f"Found columns: {rain_df.columns.tolist()}")

# Parse / cast
rain_df["Date"] = pd.to_datetime(rain_df["Date"], errors="coerce")
rain_df["Region_ID"] = pd.to_numeric(rain_df["Region_ID"], errors="coerce")
rain_df["Precip_mm"] = pd.to_numeric(rain_df["Precip_mm"], errors="coerce")

# Keep rows with keys; DO NOT drop Precip_mm missing yet (we use it to detect masked cells)
rain_df = rain_df.dropna(subset=["Date", "Region_ID"]).copy()
rain_df["Region_ID"] = rain_df["Region_ID"].astype(int)

# Clip negative precip (safe)
rain_df["Precip_mm"] = rain_df["Precip_mm"].clip(lower=0)

print("✅ Rain schema standardized:",
      rain_df[["Region_ID", "Date", "Precip_mm"]].head(3).to_string(index=False))

# =======================
# 3) DROP OCEAN/COASTAL CELLS: KEEP ONLY RAINFALL-VALID CELLS
#    (cells with at least one non-null precip value)
# =======================
valid_cells = (rain_df.groupby("Region_ID")["Precip_mm"]
               .apply(lambda s: s.notna().any()))
valid_cells = valid_cells[valid_cells].index.tolist()

print("Rainfall-valid grid cells:", len(valid_cells))
print("Example valid cells:", valid_cells[:25])

# Filter BOTH datasets to the same cell set
ls_df = ls_df[ls_df["Region_ID"].isin(valid_cells)].copy()
rain_df = rain_df[rain_df["Region_ID"].isin(valid_cells)].copy()

print("LS rows after excluding no-rain cells:", len(ls_df))
print("Rain rows after excluding no-rain cells:", len(rain_df))

# Now drop remaining missing precip rows (if any) and fill (optional)
# If you truly want to delete any date with no rainfall value:
rain_df = rain_df.dropna(subset=["Precip_mm"]).copy()

# =======================
# 4) RAIN FEATURE ENGINEERING
# =======================
rain_df = rain_df.sort_values(["Region_ID", "Date"]).reset_index(drop=True)

def add_rain_features(group):
    daily = group["Precip_mm"].shift(1)  # shift(1) prevents leakage
    out = pd.DataFrame(index=group.index)
    for w in [7, 15, 30, 60, 90]:
        out[f"rain_sum_{w}"] = daily.rolling(window=w, min_periods=1).sum()
    out["rain_days_7"] = (daily > 1.0).astype(float).rolling(window=7, min_periods=1).sum()
    return out

# avoid future pandas warning if available
try:
    rain_feats = rain_df.groupby("Region_ID", group_keys=False).apply(add_rain_features, include_groups=False)
except TypeError:
    rain_feats = rain_df.groupby("Region_ID", group_keys=False).apply(add_rain_features)

rain_eng = pd.concat([rain_df, rain_feats], axis=1)

# =======================
# 5) MERGE
# =======================
print("Merging...")
merged_df = pd.merge(ls_df, rain_eng, on=["Date", "Region_ID"], how="left")

# Delete any LS row that still failed to find rain (should be near-zero now)
before = len(merged_df)
merged_df = merged_df.dropna(subset=["Precip_mm"]).copy()
print(f"Deleted {before - len(merged_df)} rows with missing rain after merge.")

# =======================
# 6) SAVE
# =======================
merged_df.to_csv(OUT_FILE, index=False)
print("-" * 60)
print(f"✅ Saved merged dataset: {OUT_FILE}")
print(f"Merged shape: {merged_df.shape}")
print(f"Missing rain after final drop: {merged_df['Precip_mm'].isna().sum()}")
print(f"Final unique regions: {merged_df['Region_ID'].nunique()}")
print("-" * 60)

print(merged_df.head(3))


Mounted at /content/drive
Loading files...
Landsat shape: (17920, 18)
Rain shape: (560960, 5)
✅ Fix A applied: dropped 9244 rows (ValidPixCount<=0). Remaining: 8676
✅ Rain schema standardized:  Region_ID       Date  Precip_mm
         0 2001-01-01        NaN
         1 2001-01-01        NaN
         2 2001-01-01        NaN
Rainfall-valid grid cells: 47
Example valid cells: [4, 5, 6, 7, 12, 13, 14, 15, 19, 20, 21, 22, 23, 26, 27, 28, 29, 30, 31, 34, 35, 36, 37, 38, 39]
LS rows after excluding no-rain cells: 6216
Rain rows after excluding no-rain cells: 411955
Merging...
Deleted 0 rows with missing rain after merge.
------------------------------------------------------------
✅ Saved merged dataset: /content/drive/MyDrive/Flood_Analysis/final_merged_dataset_rainValidCells.csv
Merged shape: (6216, 27)
Missing rain after final drop: 0
Final unique regions: 47
------------------------------------------------------------
   Region_ID       Date     lat_x      lon_x  ValidFrac_Land  ValidPixC

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Flood_Analysis/final_merged_dataset_rainValidCells.csv")
print(df["Precip_mm"].describe())
print(df[["Precip_mm","rain_sum_7","rain_sum_30"]].quantile([0.5,0.9,0.99]))


count    6216.000000
mean       72.625765
std       131.883336
min         0.000000
25%         0.595978
50%        28.908240
75%        96.527430
max      1526.888313
Name: Precip_mm, dtype: float64
       Precip_mm   rain_sum_7   rain_sum_30
0.50   28.908240   551.301501   2397.733870
0.90  176.578507  2338.310093   9235.176691
0.99  742.985107  4652.637771  16443.349897


In [ ]:
for c in ["WaterFrac_Valid", "WaterFrac_Valid_NoPerm"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)
print((df["WaterFrac_Valid"] > 1).sum(), (df["WaterFrac_Valid_NoPerm"] > 1).sum())
print(df[["WaterFrac_Valid","WaterFrac_Valid_NoPerm"]].describe())


0 0
       WaterFrac_Valid  WaterFrac_Valid_NoPerm
count      6216.000000             6216.000000
mean          0.080102                0.080681
std           0.168349                0.168968
min           0.000000                0.000000
25%           0.000000                0.000000
50%           0.001500                0.001500
75%           0.085021                0.087249
max           1.000000                1.000000


In [ ]:
m = (df[["Region_ID","lat_x","lon_x","lat_y","lon_y"]]
     .assign(dlat=lambda x: (x.lat_x-x.lat_y).abs(),
             dlon=lambda x: (x.lon_x-x.lon_y).abs()))
print("Max centroid mismatch:", m["dlat"].max(), m["dlon"].max())


Max centroid mismatch: 0.0 0.0


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Flood_Analysis/final_merged_dataset_rainValidCells.csv")
df["Date"] = pd.to_datetime(df["Date"])
dup = df.duplicated(subset=["Region_ID","Date"]).sum()
print("Duplicate (Region_ID, Date) rows:", dup)


Duplicate (Region_ID, Date) rows: 0
